In [ ]:
repository_filter: list[str] = []
top_n_repos: int = 20

In [ ]:
import pandas as pd
from code_data_science import data_table as dt
from moderne_visualizations_misc.reusable import quality_utils as qu
import plotly.express as px
import warnings

warnings.simplefilter("ignore")

df = dt.read_csv("../samples/test_quality_issues.csv")
df = qu.filter_repos(df, repository_filter)
df = qu.add_repo_short(df)
df = qu.add_class_short(df)

In [ ]:
if len(df) == 0:
    fig = qu.empty_figure()
else:
    total_repos = df["repoShort"].nunique()
    repo_totals = df.groupby("repoShort").size().nlargest(top_n_repos)
    top_repos = repo_totals.index.tolist()
    elided = total_repos - len(top_repos)
    filtered = df[df["repoShort"].isin(top_repos)].copy()

    sev_rank = {"high": 3, "medium": 2, "low": 1}
    filtered["severity_rank"] = filtered["severity"].map(sev_rank).fillna(0)

    grouped = (
        filtered.groupby(["repoShort", "classShort", "issueType"])
        .agg(count=("severity", "size"), severity_rank=("severity_rank", "max"))
        .reset_index()
    )

    fig = px.treemap(
        grouped,
        path=["repoShort", "classShort", "issueType"],
        values="count",
        color="severity_rank",
        color_continuous_scale=["#4CAF50", "#FFE082", "#EF5350"],
        range_color=[1, 3],
        title="Test Quality Issue Treemap",
    )
    fig.update_layout(
        height=700,
        margin=dict(l=20, r=20, t=60, b=60),
        coloraxis_colorbar=dict(
            title="Severity",
            tickvals=[1, 2, 3],
            ticktext=["low", "medium", "high"],
        ),
    )
    fig.update_traces(
        hovertemplate="<b>%{label}</b><br>Count: %{value}<extra></extra>",
    )

    if elided > 0:
        fig.add_annotation(
            text=f"Showing top {len(top_repos)} of {total_repos} repositories. {elided} additional repositories with issues not shown.",
            xref="paper", yref="paper", x=0.5, y=-0.08,
            showarrow=False, font=dict(size=11, color="#888"),
        )

fig.show()